# Stage 0 — Cascade Evaluation on a Real Model (Qwen3-4B-Thinking-2507)

This notebook runs the Stage 0 cascade-evaluation harness (`quant_research.cascade`) against a real
model for the first time. Everything up to this point (`cascade.py`, `hf_runner.py`, both test
suites) was validated on synthetic/mock data only — this is the first real-model smoke test.

**What this does:**
1. Loads a small subset of [MATH-500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500) problems.
2. Generates a reference trace with **Qwen3-4B-Thinking-2507 in BF16**.
3. Teacher-forces that trace's tokens through an **NF4-quantized** copy of the same model (via
   `bitsandbytes`), and — as a sanity check on the harness itself — through the **BF16 model
   teacher-forced on its own trace**.
4. Also free-runs the NF4 candidate independently, so `propagated_to_final_answer` reflects the
   candidate's *actual* answer rather than an assumption.
5. Scores everything with `first_error_step` / `error_propagation_rate` /
   `divergence_position_distribution` from `cascade.py`.

**Scope / honesty check:** NF4 (bitsandbytes) is a calibration-free PTQ stand-in for the eventual
MR-GPTQ / NVFP4 conditions EXPERIMENTS.md's Stage 1 actually wants — Colab GPUs (T4/A100,
Ampere-class or older) have no native FP4 tensor cores, so this cannot produce a real NVFP4 result or
a like-for-like Stage 1 attribution comparison. This run is a **harness smoke test against a real
model**: does the plumbing work end-to-end and do the metrics come out sane. N is small (a handful of
problems) for the same reason — validating the pipeline, not statistical power.

**Requirements:** Colab GPU runtime (Runtime → Change runtime type → GPU). A T4 (free tier) should be
enough for a 4B model in BF16 plus a second NF4 copy; if you hit OOM, restart and run one quant
condition at a time, or use an A100 runtime.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os

if not os.path.isdir("quant-research"):
    !git clone https://github.com/ZGCompute/quant-research.git
%cd quant-research

# Use the %pip magic, not !pip — on Colab, !pip can install into a
# different environment than the one the notebook kernel is actually
# running (a long-standing Colab quirk), which silently produces exactly
# "ModuleNotFoundError: No module named 'quant_research'" in the next
# cell even though pip reports success here. %pip always targets the
# running kernel's environment.
%pip install -q -e ".[hf]"

# Verify the install actually landed before moving on, so a failure
# surfaces here with a clear message instead of a confusing
# ModuleNotFoundError several cells later.
import importlib

import quant_research

importlib.reload(quant_research)
print("quant_research installed at:", quant_research.__file__)
print(
    "If this cell fails with ModuleNotFoundError even after the fix above, "
    "go to Runtime -> Restart session, then Runtime -> Run all."
)


In [ ]:
import json
from pathlib import Path

from datasets import load_dataset

from quant_research.cascade import (
    divergence_position_distribution,
    error_propagation_rate,
    score_cascade,
)
from quant_research.hf_runner import (
    extract_boxed_answer,
    generate_reference,
    load_model,
    teacher_forced_step_distributions,
)

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
N_PROBLEMS = 8
MAX_NEW_TOKENS = 1024


## Load a small MATH-500 subset

`answer` is the dataset's ground-truth answer, used only to report whether the reference model
itself got the problem right (not part of the cascade metric, which compares candidate-vs-reference,
not candidate-vs-ground-truth).


In [ ]:
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
problems = dataset.select(range(N_PROBLEMS))
for p in problems:
    print(p["unique_id"], "-", p["problem"][:80].replace("\n", " "), "...")


## Load models

Two copies of the same checkpoint: one BF16 (reference), one NF4 (candidate). Loaded separately so
both stay resident for teacher-forced scoring; if you're on a memory-constrained runtime, comment out
one `load_model` call and run the two conditions in separate sessions instead.


In [ ]:
reference = load_model(MODEL_ID, quant_mode="bf16")
candidate = load_model(MODEL_ID, quant_mode="nf4")


## Build a thinking-mode prompt


In [ ]:
def build_prompt(problem: str) -> str:
    messages = [{
        "role": "user",
        "content": problem + "\n\nPlease reason step by step, and put your final answer within \\boxed{}.",
    }]
    return reference.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
    )


## Run Stage 0

For each problem:
1. Generate the reference (BF16) trace freely.
2. Teacher-force the reference model on its *own* trace as a harness sanity check — this should
   diverge almost nowhere; if it doesn't, that's a bug in the scoring path, not a quantization effect.
3. Teacher-force the NF4 candidate on the reference trace to find where its own next-token
   distribution first diverges from the reference (`first_error_step`).
4. Free-run the NF4 candidate independently to get its actual final answer, so
   `propagated_to_final_answer` reflects a real outcome rather than an assumption.


In [ ]:
records = []
cascade_results = []
trace_lengths = []

for problem in problems:
    prompt = build_prompt(problem["problem"])

    ref_token_ids, ref_text = generate_reference(reference, prompt, max_new_tokens=MAX_NEW_TOKENS)
    ref_answer = extract_boxed_answer(ref_text)

    self_check_steps = teacher_forced_step_distributions(reference, prompt, ref_token_ids)
    self_check_result = score_cascade(ref_token_ids, self_check_steps, ref_answer, ref_answer)

    candidate_steps = teacher_forced_step_distributions(candidate, prompt, ref_token_ids)

    cand_token_ids, cand_text = generate_reference(candidate, prompt, max_new_tokens=MAX_NEW_TOKENS)
    cand_answer = extract_boxed_answer(cand_text)

    candidate_result = score_cascade(ref_token_ids, candidate_steps, ref_answer, cand_answer)

    cascade_results.append(candidate_result)
    trace_lengths.append(len(ref_token_ids))
    records.append({
        "unique_id": problem["unique_id"],
        "level": problem["level"],
        "trace_length": len(ref_token_ids),
        "reference_answer": ref_answer,
        "dataset_answer": problem["answer"],
        "reference_correct": ref_answer == problem["answer"],
        "self_check_diverged": self_check_result.diverged,
        "self_check_first_error_step": self_check_result.first_error_step,
        "candidate_answer": cand_answer,
        "candidate_diverged": candidate_result.diverged,
        "candidate_first_error_step": candidate_result.first_error_step,
        "candidate_divergence_margin": candidate_result.divergence_margin,
        "candidate_divergence_entropy": candidate_result.divergence_entropy,
        "candidate_propagated_to_final_answer": candidate_result.propagated_to_final_answer,
    })

    print(
        f"{problem['unique_id']}: trace_len={len(ref_token_ids)} "
        f"self_check_diverged={self_check_result.diverged} "
        f"candidate_diverged={candidate_result.diverged} "
        f"first_error_step={candidate_result.first_error_step} "
        f"propagated={candidate_result.propagated_to_final_answer}"
    )


## Aggregate and save


In [ ]:
self_check_divergence_rate = sum(r["self_check_diverged"] for r in records) / len(records)
print(f"self-check divergence rate (should be ~0): {self_check_divergence_rate:.3f}")
print(f"candidate error-propagation rate: {error_propagation_rate(cascade_results):.3f}")

positions = divergence_position_distribution(cascade_results, trace_lengths)
print(f"candidate divergence positions (0=start of trace, 1=end): {positions}")


In [ ]:
output_path = Path("stage0_results.json")
output_path.write_text(json.dumps({
    "model_id": MODEL_ID,
    "n_problems": N_PROBLEMS,
    "records": records,
    "self_check_divergence_rate": self_check_divergence_rate,
    "error_propagation_rate": error_propagation_rate(cascade_results),
    "divergence_positions": positions,
}, indent=2))
print(f"Saved to {output_path.resolve()}")

# Optional: persist to Drive so results survive the Colab session ending.
# from google.colab import drive
# drive.mount('/content/drive')
# output_path.rename('/content/drive/MyDrive/stage0_results.json')


## Next steps

- If `self_check_diverged` is True for more than a handful of steps, treat that as a harness bug to
  fix (tokenizer/dtype mismatch in the scoring path) before trusting the candidate-vs-reference
  numbers — the BF16 model teacher-forced on its own greedy trace should be ~deterministic.
- Once this smoke test looks sane, scale `N_PROBLEMS` up and add AIME 2025/2026 + GPQA-Diamond, per
  EXPERIMENTS.md Stage 0's full benchmark list, not just MATH-500.
- Stage 1 (error attribution: weight vs. KV-cache vs. activation) needs a genuinely like-for-like
  comparison — bitsandbytes NF4 here is not that; it conflates weight quantization with
  bitsandbytes-specific compute-path effects. Stage 1 will need MR-GPTQ/NVFP4-style conditions, which
  needs Blackwell-class hardware not available on Colab (see the hardware roadmap in project memory).
